# EDA — Exploratory Data Analysis
## Dataset: Fruit and Vegetable Image Recognition (Kaggle)

Notebook này thực hiện phân tích khám phá dữ liệu trước khi huấn luyện model, theo đúng yêu cầu môn học:

- Tổng số ảnh, tổng số class, tên từng class
- Số lượng ảnh của từng class (train/validation/test)
- Phân bố dữ liệu giữa các class (class imbalance)
- Kích thước ảnh, định dạng ảnh
- Ảnh mẫu của từng class
- Phát hiện ảnh lỗi/bất thường

**Yêu cầu trước khi chạy notebook này:** dataset đã được tải về `data/raw/` theo cấu trúc `train/`, `validation/`, `test/` — xem `TRAINING_GUIDE.md` hoặc `scripts/download_dataset.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt

from configs.config import CONFIG
from src.data.dataset import discover_dataset
from src.data.eda import (
    class_distribution_df,
    plot_class_distribution,
    detect_class_imbalance,
    sample_images_grid,
    image_size_stats,
    plot_image_size_distribution,
)

%matplotlib inline

## 1. Khám phá cấu trúc dataset thực tế trên đĩa

Không giả định trước số class/số ảnh — đọc trực tiếp từ thư mục `data/raw/`.

In [ ]:
splits, info = discover_dataset(CONFIG)

print(f"Tổng số class: {info.num_classes}")
print(f"Tổng số ảnh (train+val+test): {info.total_images}")
print(f"\nTên các class:\n{info.class_names}")

In [ ]:
for split, counts in info.counts_per_split.items():
    print(f"{split}: {sum(counts.values())} ảnh, {len(counts)} class")

## 2. Class distribution — số lượng ảnh mỗi class

In [ ]:
df = pd.DataFrame(class_distribution_df(info))
pivot = df.pivot(index="class", columns="split", values="count").fillna(0).astype(int)
pivot

In [ ]:
fig = plot_class_distribution(info, split="train")
plt.show()

## 3. Class imbalance

In [ ]:
imbalance = detect_class_imbalance(info, split="train")
imbalance

**Nhận xét:** nếu `is_imbalanced=True`, cần cân nhắc: dùng `class_weight` khi train, hoặc oversampling/augmentation mạnh hơn cho class ít ảnh, và **không đánh giá model chỉ bằng Accuracy** (macro F1 sẽ phản ánh đúng hơn khi có imbalance).

## 4. Ảnh mẫu của từng class

In [ ]:
fig = sample_images_grid(splits["train"], info.class_names)
plt.show()

## 5. Kích thước & định dạng ảnh

In [ ]:
sizes, corrupted = image_size_stats(splits["train"])
print(f"Số ảnh đã kiểm tra (mẫu): {len(sizes)}")
print(f"Số kích thước khác nhau: {len(set(sizes))}")
print(f"Số ảnh lỗi/không đọc được (trong mẫu): {len(corrupted)}")
if corrupted:
    print("Danh sách ảnh lỗi:", corrupted)

In [ ]:
fig = plot_image_size_distribution(sizes)
plt.show()

## 6. Kết luận EDA

Điền vào đây sau khi chạy notebook trên dataset thật:

- Tổng số class: ...
- Tổng số ảnh: ...
- Có class imbalance không? ...
- Kích thước ảnh phổ biến: ...
- Có ảnh lỗi/bất thường không? ...
- Quyết định preprocessing: resize về bao nhiêu? có cần augmentation mạnh không (vì sao)?